<a href="https://colab.research.google.com/github/accoelhos/Trabalho_Teoria_Computacao/blob/main/trabalho.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Definição do Problema

## 1.1 Informações Importantes nos Estados

Os estados do autômato precisam rastrear


*   Contexto base: se ele está lendo algo decimal (0-9), haxadecimal (A-F) ou científico.
*   Estágio da parte numérica: se o dígito que ele leu pertence à parte principal do número, ao expoente (em caso de científico) ou à parte imaginária (em caso de número complexo).
* Flags de símbolos: deve ter conhecimento de prefixos como 0x e #, sufixos como h, i e separadores como ., e, E.

## 1.2 Representação como linguagem L

A linguagem do autômato é a união de cinco sublinguagens

$$L = L_{int} \cup L_{real} \cup L_{sci} \cup L_{hex} \cup L_{comp}$$

Com o alfabeto $\Sigma = \{0..9, A..F, a..f, ., +, -, e, E, x, h, i, \#\}$.

## 1.3 Estratégia para solução do problema

1. Desenvolver os autômatos para cada umas das notações possíveis separadamente

2. Fazer o produto de todos esses autômatos levando em consideração casos de duplicidade de significado como o de E/e para notação científica e hexadecimal e o 0 para sufixo de hexadecimal e como número.

3. Implementar no código para fazer testes automatizados

4. Em caso de erro, tomar esses passos em ordem e caso o primeiro resolva não é necessário seguir adiante:
  - Verificar se há erros no código
  - Verificar se há erros no autômato produto
  - Verificar se há erros no autômato da notação que falhou no teste

# 2. Definição Formal do AF

## 2.1 Descrição dos estados ($Q$)

| Estado | Propósito | Categoria de Aceitação |
| :--- | :--- | :--- |
| **q0** | **Estado Inicial**: Ponto de partida da leitura. | - |
| **q1** | **Número**: Após leitura de número, excluindo o "0" | **Inteiros** |
| **q2** | **Sinal**: Lendo sinal (+ ou -). | - |
| **q3** | **Número 0**: Após leitura de "0" inicial. | **Número "0"** |
| **q4** | **Ponto**: Após leitura de "." inicial (ex.: .5). | - |
| **q5** | **Ponto após número**: Estado final de números reais, ou continuação de números complexos | **Reais** |
| **q6** | **Sinal da Parte Imaginária**: Após + ou - dentro da parte imaginária. | - |
| **q7** | **Parte Imaginária**: Lendo dígitos da parte imaginária. | - |
| **q8** | **Números da Parte Imaginária**: Permite colocar diversos números na parte imaginária de um número complexo. | - |
| **q9** | **Ponto da Parte Imaginária**: Após ler "." na parte imaginária de um número complexo. | - |
| **q10** | **Complexo**: Após validar o sufixo "i" da parte imaginária. | **Complexos** |
| **q11** | **Números**: Após número com sinal. | **Inteiros** |
| **q12** | **Ponto**: Valida o ponto após número com sinal | **Reais** |
| **q13** | **Prefixo do Hexa**: Valida o prefixo "0x" ou "#" de números hexadecimais. | - |
| **q14** | **Hexadecimal**: Valida números hexadecimais com prefixo. | **Hexadecimais com prefixo** |
| **q15** | **Sufixo Hexadecimal**: Valida o sufixo de um número hexadecimal. | **Hexadecimal com sufixo** |
| **q16** | **Euler**: Após validar o "e" de hexadecimal ou científico. | **Hexadecimal com 1 e** |
| **q17** | **Número após e**: Valida números após o e. | **Hexadecimal ou científico** |
| **q18** | **Sinal expoente**: Sinal do expoente de números científicos. | - |
| **q19** | **Número expoente**: Número do expoente de números científicos. | **Científicos** |
| **q20** | **Euler**: Valida euler após números reais. | - |
| **q21** | **Número** : Valida número após a vírgula de números complexos. | - |
| **q22** | **Hexa com sufixo** : Valida o fluxo de hexadecimal com sufixo. | - |
| **qerr** | **Erro**: Estado de rejeição para sequências inválidas. | - |

## 2.2 Alfabeto de entrada ($\Sigma$)

$$\Sigma = \{0..9, A..F, a..f, ., +, -, e, E, x, h, i, \#\}$$

### 2.3 Descrição da Função de Transição ($\delta$)
Abaixo, a tabela de transição que mapeia o comportamento do autômato.

*Foram utilizadas as seguintes convenções:*

- `n` → dígitos `0–9`
- `0` → prefixo `0`
- `s` → `{+, -}`
- `e` → `{e, E}`
- `l` → `{A–F, a–f}`
- `i` → `i`
- `h` → `h`
- `#` → `#`
- `x` → `x`


| Estado Atual | Símbolo (σ) | Próximo Estado | Descrição |
|--------------|------------|----------------|----------|
| **q0** | n | q1 | Início com dígito 1–9 |
| **q0** | 0 | q3 | Início com zero |
| **q0** | s | q2 | Sinal inicial |
| **q0** | . | q4 | Número começando com ponto |
| **q0** | # | q13 | Prefixo hexadecimal (#) |
| **q0** | l | q22 | Início direto como hexadecimal |
| **q0** | e | q22 | Letra tratada como hexadecimal |
| **q1** | n | q1 | Loop da parte inteira |
| **q1** | . | q5 | Parte decimal |
| **q1** | e | q16 | Início do expoente |
| **q1** | s | q6 | Possível número complexo |
| **q1** | h | q15 | Sufixo hexadecimal |
| **q1** | l | q22 | Hexadecimal |
| **q2** | n | q11 | Dígitos após sinal |
| **q3** | n | q1 | Continua número |
| **q3** | x | q13 | Prefixo 0x |
| **q3** | . | q5 | Vai para float |
| **q3** | l | q14 | Hexadecimal |
| **q3** | e | q16 | Número Científico iniciado em 0 |
| **q4** | n | q5 | Parte fracionária |
| **q5** | n | q5 | Loop decimal |
| **q5** | e | q20 | Expoente |
| **q5** | s | q6 | Complexo |
| **q6** | n | q7 | Parte imaginária |
| **q7** | n | q8 | Continua número |
| **q7** | . | q9 | Parte decimal do imaginário |
| **q7** | i | q10 | Final complexo |
| **q8** | n | q8 | Loop inteiro |
| **q8** | . | q21 | Decimal |
| **q8** | i | q10 | Final complexo |
| **q9** | n | q21 | Continua decimal |
| **q11** | n | q11 | Loop inteiro |
| **q11** | . | q12 | Decimal |
| **q11** | e | q20 | Científico |
| **q12** | n | q12 | Loop decimal |
| **q13** | n, l, e | q14 | Corpo hexadecimal |
| **q14** | n, l, e | q14 | Loop hexadecimal |
| **q16** | n | q17 | Expoente direto |
| **q16** | s | q18 | Sinal do expoente |
| **q16** | l, e | q22 | Hexadecimal |
| **q16** | h | q15 | Sufixo Hexadecimal |
| **q17** | n | q17 | Loop expoente |
| **q17** | l, e | q22 | Hexadecimal |
| **q17** | h | q15 | Final hexadecimal com sufixo |
| **q18** | n | q19 | Expoente após sinal |
| **q19** | n | q19 | Loop expoente |
| **q20** | n | q19 | Expoente |
| **q20** | s | q18 | Permite Sinal após euler |
| **q21** | n, i | q10 | Final complexo |

### 2.4 Estados Finais ($F$)
O conjunto de estados de aceitação é definido por:
$$F = \{q_1, q_3, q_5, q_{10}, q_{11}, q_{12}, q_{14}, q_{15}, q_{17}, q_{19}\}$$

# 3. Implementação

- Código completo da classe ```FiniteAutomata```
- Funções auxiliares



In [6]:
from typing import Dict, Tuple, Optional

#Definição do alfabeto
#Obs.: Apenas o num, sinal, euler e letra são utilizados no código, os outros são para visibilidade do usuário
# 0 não está em num e nem E/e em letra pois são considerados como símbolos a parte já que podem ser ambíguos
num = ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]
sinal = ["-", "+"]
ponto = ['.']
img = ['i']
euler = ['E', 'e']
letra = ['A', 'B', 'C', 'D', 'F', 'a', 'b', 'c', 'd', 'f']
p1 = ['0s']
p2 = ['x']
p3 = ['#']
sufixo = ['h']

class FiniteAutomata:
  delta: Dict[Tuple[str, str], str] = None

  def __init__(self, tape):
    # Inicialização

    #Tape é a fita que será analisada
    self.tape = tape

    #Head é a posição do caractere na fita analisada
    self.head = 0

    #Estado_atual para validar com as funções de transição e estado final
    self.estado_atual = 'q0'

    #Definição das funções de transição
    self.delta = {
        ('q0', 'n'): 'q1',
        ('q0', 's'): 'q2',
        ('q0', 'p1'): 'q3',
        ('q0', '.'): 'q4',
        ('q0', '#'): 'q13',
        ('q0', 'l'): 'q22',
        ('q0', 'e'): 'q22',

        ('q1', 'n'): 'q1',
        ('q1', '.'): 'q5',
        ('q1', 'e'): 'q16',
        ('q1', 's'): 'q6',
        ('q1', 'h'): 'q15',
        ('q1', 'l'): 'q22',

        ('q2', 'n'): 'q11',

        ('q3', 'n'): 'q1',
        ('q3', 'x'): 'q13',
        ('q3', '.'): 'q5',
        ('q3', 'l'): 'q22',
        ('q3', 'e'): 'q16',

        ('q4', 'n'): 'q5',

        ('q5', 'n'): 'q5',
        ('q5', 'e'): 'q20',
        ('q5', 's'): 'q6',

        ('q6', 'n'): 'q7',

        ('q7', 'n'): 'q8',
        ('q7', '.'): 'q9',
        ('q7', 'i'): 'q10',

        ('q8', 'n'): 'q8',
        ('q8', '.'): 'q21',
        ('q8', 'i'): 'q10',

        ('q9', 'n'): 'q21',

        ('q11', 'n'): 'q11',
        ('q11', '.'): 'q12',
        ('q11', 'e'): 'q20',

        ('q12', 'n'): 'q12',

        ('q13', 'n'): 'q14',
        ('q13', 'l'): 'q14',
        ('q13', 'e'): 'q14',

        ('q14', 'n'): 'q14',
        ('q14', 'l'): 'q14',
        ('q14', 'e'): 'q14',

        ('q16', 'n'): 'q17',
        ('q16', 'l'): 'q22',
        ('q16', 'e'): 'q22',
        ('q16', 's'): 'q18',
        ('q16', 'h'): 'q15',

        ('q17', 'n'): 'q17',
        ('q17', 'l'): 'q22',
        ('q17', 'e'): 'q22',
        ('q17', 'h'): 'q15',

        ('q18', 'n'): 'q19',

        ('q19', 'n'): 'q19',

        ('q20', 'n'): 'q19',
        ('q20', 's'): 'q18',

        ('q21', 'n'): 'q10',
        ('q21', 'i'): 'q10',

        ('q22', 'n'): 'q22',
        ('q22', 'l'): 'q22',
        ('q22', 'e'): 'q22',
        ('q22', 'h'): 'q15',


    }

    #Definição dos estados finais e do estado de erro
    self.estado_final = {'q1', 'q3', 'q5', 'q10', 'q11', 'q12', 'q14', 'q15', 'q17', 'q19'}
    self.estado_erro   = 'qerr'

  def step(self):
      """
      Executa um passo do AF
      Retorna: keep_running
      """

      try:
        self.tape = str(self.tape)
      except E:
        print('A sequencia não pode ser convertida em string')
        self.estado_atual = 'qerr'
        return False

      #Unifica símbolos que pertencem ao mesmo significado
      simbolo = self.tape[self.head]
      if self.head == 0 and simbolo == "0":
        simbolo = 'p1'
      if simbolo in num:
        simbolo = 'n'
      if simbolo in letra:
        simbolo = 'l'
      if simbolo in euler:
        simbolo = 'e'
      if simbolo in sinal:
        simbolo = 's'

      transicao = (self.estado_atual, simbolo)

      #Verifica se o estado com o símbolo são válidos, se não é erro e retorna false
      self.estado_atual = self.delta.get(transicao, self.estado_erro)
      self.head += 1

      keep_running = (self.head < len(self.tape) and self.estado_atual != self.estado_erro)
      return keep_running

  def execute(self):
      """
      Executa o AF para a fita fornecida
      Retorna: (accept)
      """

      #Verifica a fita caractere por caractere até o final e então verifica se está no estado final, se estiver é True, se não False
      if len(self.tape) <= 0:
        return False

      while self.step():
        pass

      accept = self.estado_atual in self.estado_final

      return accept

# 4. Testes

- Casos pequenos
- Verificação de correção


In [7]:
# Testes unitários por notação, válidos e inválidos
casos = [
    #Inteiros

    #--Válidos
    ("123",       True),
    ("-456",      True),
    ("+789",      True),
    ("0",         True),

    #--Invalidos
    ("",          False),   # fita vazia
    ("+",         False),   # só sinal, sem dígito
    ("--5",       False),   # dois sinais
    ("3+",       False),
    ("01a", False),
    ("++1", False),
    ("-+1", False),
    ("1-", False),

    # Reais

    #--Válidos
    ("0.", True),
    (".0", True),
    ("000.000", True),
    ("3.14",      True),
    ("-0.5",      True),
    ("+2.0",      True),
    (".5",        True),
    ("5.",        True),

    #--Invalidos
    (".", False),
    ("..5", False),
    ("5..0", False),
    ("5.0.1", False),
    (".e1", False),

    #Notação Científica

    #--Válidos
    ("1.23e4",    True),
    ("-5E-10",    True),
    ("6e+7",      True),
    (".5e3",      True),
    ("2.e-2",     True),
    ("1e0", True),
    ("1E+0", True),
    ("1e-0", True),
    ("0e0", True),

    #--Inválidos
    ("1e+", False),
    ("1e-", False),
    ("1e1.5", False),      # expoente não pode ser float

    #Hexadecimal

    #--Válidos
    ("0xFF",      True),
    ("0x1A3",     True),
    ("#FF",       True),
    ("FFh",       True),
    ("0x0", True),
    ("0xabc", True),
    ("#1A", True),
    ("ABCDEFh", True),
    ("eh", True),

    #--Inválidos
    ("0x", False),
    ("#", False),
    ("0xG", False),        # G não é hex
    ("0x1.2", False),      # ponto inválido
    ("h", False),
    ("FFhh", False),
    ("e10", False),        # sem base
    ("1e", False),         # sem expoente

    #Complexos

    #--Válidos
    ("3+4i",      True),
    ("2.5-1.3i",  True),

    #--Inválidos
    ("i", False),          # você NÃO definiu imaginário puro
    ("3+i", False),        # falta número após +
    ("3+4", False),        # sem i
    ("3+4ii", False),
    ("3++4i", False),
    ("3+4i5", False),

    #Casos misturados
    ("000e10", True),
    (".0e0", True),
    ("0.e1", True),

    ("0x1e", True),
    ("0x1e+2", False),

    ("3+4e2i", False),
    ("3+4.0e2i", False),

    ("1.2.3", False),
    ("0x1e10", True),
    ("123h", True),
    ("123he", False),
    ("3.14i", False),

]

for fita, esperado in casos:
    af      = FiniteAutomata(fita)
    result  = af.execute()
    status  = "Acertou" if result == esperado else "Errou"
    print(f"{status}  '{fita!s:<8}' → {'aceite' if result else 'rejeita'}  (esperado: {'aceite' if esperado else 'rejeita'})")

Acertou  '123     ' → aceite  (esperado: aceite)
Acertou  '-456    ' → aceite  (esperado: aceite)
Acertou  '+789    ' → aceite  (esperado: aceite)
Acertou  '0       ' → aceite  (esperado: aceite)
Acertou  '        ' → rejeita  (esperado: rejeita)
Acertou  '+       ' → rejeita  (esperado: rejeita)
Acertou  '--5     ' → rejeita  (esperado: rejeita)
Acertou  '3+      ' → rejeita  (esperado: rejeita)
Acertou  '01a     ' → rejeita  (esperado: rejeita)
Acertou  '++1     ' → rejeita  (esperado: rejeita)
Acertou  '-+1     ' → rejeita  (esperado: rejeita)
Acertou  '1-      ' → rejeita  (esperado: rejeita)
Acertou  '0.      ' → aceite  (esperado: aceite)
Acertou  '.0      ' → aceite  (esperado: aceite)
Acertou  '000.000 ' → aceite  (esperado: aceite)
Acertou  '3.14    ' → aceite  (esperado: aceite)
Acertou  '-0.5    ' → aceite  (esperado: aceite)
Acertou  '+2.0    ' → aceite  (esperado: aceite)
Acertou  '.5      ' → aceite  (esperado: aceite)
Acertou  '5.      ' → aceite  (esperado: aceite)
Acer

# 5. Demonstração Formal de Corretude

## 5.1 Definição Formal do AFD Implementado

O AFD implementado é a 5-tupla $M = (Q, \Sigma, \delta, q_0, F)$ onde:

$$Q = \{q_0, q_1, q_2, q_3, q_4, q_5, q_6, q_7, q_8, q_9, q_{10}, q_{11}, q_{12}, q_{13}, q_{14}, q_{15}, q_{16}, q_{17}, q_{18}, q_{19}, q_{20}, q_{21}, q_{22}, q_{err}\}$$

$$\Sigma_{raw} = \{0\text{-}9,\ A\text{-}F,\ a\text{-}f,\ .,\ +,\ -,\ e,\ E,\ x,\ h,\ i,\ \#\}$$

$$F = \{q_1, q_3, q_5, q_{10}, q_{11}, q_{12}, q_{14}, q_{15}, q_{17}, q_{19}\}$$

### Função de classificação $\gamma : \Sigma_{raw} \to \Sigma_{classe}$

Antes de consultar $\delta$, cada símbolo $\sigma$ é mapeado pela função $\gamma$:

| Símbolo original | Classe | Justificativa |
|:---|:---|:---|
| $1$–$9$ | $n$ | Dígitos não-zero |
| $0$ | $0$ | Considerado uma transição separada se estiver no início da fita $0x$ |
| $A$–$D, F, a$–$d, f$ | $l$ | Letras hexadecimais (exceto $e$) |
| $e, E$ | $e$ | Ambíguo: euler (científico) ou dígito hex |
| $+, -$ | $s$ | Sinais |
| $.$ | $.$ | Separador decimal |
| $x$ | $x$ | Prefixo hexadecimal |
| $h$ | $h$ | Sufixo hexadecimal |
| $i$ | $i$ | Sufixo imaginário |
| $\#$ | $\#$ | Prefixo hexadecimal alternativo |

### Descrição semântica dos estados

| Estado | Papel no autômato |
|:---|:---|
| $q_0$ | Inicial |
| $q_1$ | Dígitos da parte inteira (sem sinal, ou após $0d^+$). **Aceita**: inteiros |
| $q_2$ | Após sinal inicial ($+$ ou $-$) |
| $q_3$ | Após leitura de $0$ isolado (bifurcação: $0x$, $0.$, $0d$, hex). **Aceita**: inteiro "$0$" |
| $q_4$ | Após ponto decimal inicial ($.d$...) |
| $q_5$ | Dígitos fracionários (após $.$ com pelo menos um dígito, ou $d^+.d^*$). **Aceita**: reais |
| $q_6$ | Após sinal da parte imaginária do complexo |
| $q_7$ | Primeiro dígito da parte imaginária (aguarda $i$, $.$ ou mais dígitos) |
| $q_8$ | Dígitos inteiros da parte imaginária (loop) |
| $q_9$ | Após ponto decimal na parte imaginária |
| $q_{10}$ | Após sufixo $i$. **Aceita**: complexos |
| $q_{11}$ | Dígitos inteiros após sinal (caminho com sinal). **Aceita**: inteiros com sinal |
| $q_{12}$ | Dígitos fracionários após sinal + ponto. **Aceita**: reais com sinal |
| $q_{13}$ | Após prefixo hex ($0x$ ou $\#$) |
| $q_{14}$ | Dígitos hexadecimais (loop). **Aceita**: hexadecimais |
| $q_{15}$ | Após sufixo $h$. **Aceita**: hexadecimais com sufixo |
| $q_{16}$ | Após $e$/$E$ vindo de $q_1$ (ambíguo: científico ou hex). **Aceita**: hex $de$ |
| $q_{17}$ | Dígitos do expoente científico (vindo de $q_{16}$ sem letras hex). **Aceita**: notação científica |
| $q_{18}$ | Após sinal do expoente ($e[+-]$) |
| $q_{19}$ | Dígitos do expoente científico (caminho geral). **Aceita**: notação científica |
| $q_{20}$ | Após $e$/$E$ vindo de parte fracionária ($q_5$) ou inteiro com sinal ($q_{11}$) |
| $q_{21}$ | Dígitos fracionários da parte imaginária (aguardando $i$) |
| $q_{22}$ | Dígitos Hexadecimais com sufixo (aguardando $h$) |
| $q_{err}$ | Estado de rejeição (absorvente) |

---

## 5.2 Definição da Linguagem $L$

A linguagem que o autômato deve reconhecer é:

$$L = L_{int} \cup L_{real} \cup L_{sci} \cup L_{hex} \cup L_{comp}$$

Onde, usando $d$ para dígitos $\{0\text{-}9\}$, $h$ para dígitos hex $\{0\text{-}9, A\text{-}F, a\text{-}f\}$, e $s$ para sinais $\{+,-\}$:

$$L_{int} = \{[s]\ d^+ \}$$
Exemplos: $123$, $-456$, $+789$, $0$

$$L_{real} = \{[s]\ (d^+.d^* \mid d^*.d^+) \}$$
Exemplos: $3.14$, $-0.5$, $.5$, $5.$

$$L_{sci} = \{r\ (e|E)\ [s]\ d^+ \mid r \in L_{int} \cup L_{real}\}$$
Exemplos: $1.23e4$, $-5E{-}10$, $.5e3$

$$L_{hex} = \{0x\ h^+ \} \cup \{\#\ h^+\} \cup \{h^+\ h_{sufixo}\}$$
Exemplos: $0xFF$, $\#1A$, $FFh$, $e10$

$$L_{comp} = \{a\ s\ b\ i \mid a \in L_{int} \cup L_{real} \text{ (sem sinal)},\ b = d^+[.d^*] \text{ ou } [d^*.]d^+\}$$
Exemplos: $3+4i$, $2.5{-}1.3i$

---

## 5.3 Prova de Corretude (Soundness): $L(M) \subseteq L$

**Método**: Para cada estado final $q_f \in F$, demonstramos que toda cadeia $w$ que leva $M$ a $q_f$ pertence a alguma sublinguagem $L_i$.

### Caso 1: Aceitação em $q_1$ — Inteiros sem sinal

**Caminhos possíveis até $q_1$:**
- $q_0 \xrightarrow{n} q_1$
- $q_3 \xrightarrow{n|0} q_1$ (após ler $0$ inicial)
- $q_1 \xrightarrow{n|0} q_1$ (loop)

**Forma da cadeia**: $[0]\ d^+$ onde $d \in \{0\text{-}9\}$. Como não há sinal, esta é da forma $d^+$, logo $w \in L_{int}$. $\square$

### Caso 2: Aceitação em $q_3$ — Zero isolado

**Único caminho**: $q_0 \xrightarrow{0} q_3$.

**Forma da cadeia**: "$0$", que satisfaz $d^+$ com um dígito, logo $w \in L_{int}$. $\square$

### Caso 3: Aceitação em $q_{11}$ — Inteiros com sinal

**Caminhos possíveis até $q_{11}$:**
- $q_0 \xrightarrow{s} q_2 \xrightarrow{n|0} q_{11}$
- $q_{11} \xrightarrow{n|0} q_{11}$ (loop)

**Forma da cadeia**: $s\ d^+$ onde $s \in \{+,-\}$, logo $w \in L_{int}$. $\square$

### Caso 4: Aceitação em $q_5$ — Reais (parte fracionária)

**Caminhos possíveis até $q_5$:**
- $q_0 \xrightarrow{.} q_4 \xrightarrow{n|0} q_5$ → cadeia $.d^+$
- $q_1 \xrightarrow{.} q_5$ → cadeia $d^+.$
- $q_3 \xrightarrow{.} q_5$ → cadeia $0.$
- $q_5 \xrightarrow{n|0} q_5$ (loop) → acumula dígitos fracionários

**Forma geral**: $[0|d^+]\ .\ d^*$ ou $.\ d^+$, que são da forma $d^+.d^*$ ou $d^*.d^+$, logo $w \in L_{real}$. $\square$

### Caso 5: Aceitação em $q_{12}$ — Reais com sinal

**Caminho**: (caminho até $q_{11}$) $\xrightarrow{.} q_{12} \xrightarrow{n|0}^* q_{12}$.

**Forma da cadeia**: $s\ d^+.d^*$, logo $w \in L_{real}$. $\square$

### Caso 8: Aceitação em $q_{17}$ — Notação científica (expoente via $q_1$)

**Caminhos possíveis até $q_{17}$:**
- $q_{16} \xrightarrow{n|0} q_{17}$
- $q_{17} \xrightarrow{n|0} q_{17}$ (loop)

$q_{16}$ é alcançado por $q_1 \xrightarrow{e} q_{16}$, portanto a forma é $d^+\ e\ d^+$.

Isso satisfaz $L_{sci}$ com parte inteira $d^+$, indicador $e$, e expoente $d^+$, logo $w \in L_{sci}$. $\square$

### Caso 9: Aceitação em $q_{19}$ — Notação científica (expoente via $q_5$, $q_{11}$ ou com sinal)

**Caminhos possíveis até $q_{19}$:**
- (caminho até $q_5$ ou $q_{11}$) $\xrightarrow{e} q_{20} \xrightarrow{n|0} q_{19}$
- (caminho até $q_{16}$) $\xrightarrow{s} q_{18} \xrightarrow{n|0} q_{19}$
- $q_{20} \xrightarrow{s} q_{18} \xrightarrow{n|0} q_{19}$
- $q_{19} \xrightarrow{n|0} q_{19}$ (loop)

**Forma geral**: $r\ (e|E)\ [s]\ d^+$ onde $r \in L_{int} \cup L_{real}$, logo $w \in L_{sci}$.

**Verificação de que múltiplos sinais são impossíveis**: $q_{20} \xrightarrow{s} q_{18}$ e $q_{18}$ **não** tem transição por $s$, portanto apenas um sinal é permitido no expoente. $\square$

### Caso 10: Aceitação em $q_{14}$ — Hexadecimal

**Caminhos possíveis até $q_{14}$:**
- $q_0 \xrightarrow{\#} q_{13} \xrightarrow{h} q_{14}$ → cadeia $\#h^+$
- $q_{13} \xrightarrow{h} q_{14}$ (após $0x$ ou $\#$)
- $q_{14} \xrightarrow{n|0|l|e} q_{14}$ (loop)

Em todos os casos, a cadeia é uma sequência de dígitos hexadecimais, opcionalmente com prefixo $0x$ ou $\#$, logo $w \in L_{hex}$. $\square$

### Caso 11: Aceitação em $q_{15}$ — Hexadecimal com sufixo $h$

**Caminhos possíveis até $q_{15}$:**
- (caminho até $q_1$ ou $q_{17}$) $\xrightarrow{h} q_{15}$.
- $q_0 \xrightarrow{n} q_{1} \xrightarrow{l} q_{22} \xrightarrow{h} q_{15}$ → cadeia $h^+h$
- $q_0 \xrightarrow{n} q_{1} \xrightarrow{e} q_{16} \xrightarrow{h} q_{15}$ → cadeia $h^+h$

A cadeia tem a forma $h^+h_{sufixo}$, logo $w \in L_{hex}$. $\square$

### Caso 12: Aceitação em $q_{10}$ — Números complexos

**Caminhos possíveis até $q_{10}$:**

1. **Parte imaginária inteira**: (parte real) $\xrightarrow{s} q_6 \xrightarrow{n|0} q_7 \xrightarrow{i} q_{10}$
2. **Parte imaginária com mais dígitos**: ... $q_7 \xrightarrow{n} q_8 \xrightarrow{n|0}^* q_8 \xrightarrow{i} q_{10}$
3. **Parte imaginária com ponto e fração**: ... $q_7 \xrightarrow{.} q_9 \xrightarrow{n|0} q_{21} \xrightarrow{i} q_{10}$ ou $q_8 \xrightarrow{.} q_{21} \xrightarrow{i} q_{10}$

**Origem da parte real**: A transição $\xrightarrow{s} q_6$ parte de $q_1$, $q_5$ ou $q_{11}$, que correspondem a inteiros ou reais válidos (sem sinal ou com sinal).

**Forma geral**: $a\ s\ b\ i$ onde:
- $a$ é a parte real (inteiro ou real válido)
- $s \in \{+,-\}$
- $b$ é a parte imaginária (inteiro ou real positivo sem sinal)
- $i$ é o sufixo imaginário

Logo $w \in L_{comp}$. $\square$

---

## 5.4 Prova de Completude: $L \subseteq L(M)$

Para cada sublinguagem, demonstramos que toda cadeia válida é aceita.

### Para $L_{int}$: cadeia $[s]\ d^+$

| Forma | Caminho de estados | Estado final |
|:---|:---|:---|
| $d^+$ (sem sinal, $d_1 \neq 0$) | $q_0 \xrightarrow{n} q_1 \xrightarrow{n|0}^* q_1$ | $q_1 \in F$ ✓ |
| $0$ | $q_0 \xrightarrow{0} q_3$ | $q_3 \in F$ ✓ |
| $0d^+$ | $q_0 \xrightarrow{0} q_3 \xrightarrow{n|0} q_1 \xrightarrow{n|0}^* q_1$ | $q_1 \in F$ ✓ |
| $sd^+$ (com sinal) | $q_0 \xrightarrow{s} q_2 \xrightarrow{n|0} q_{11} \xrightarrow{n|0}^* q_{11}$ | $q_{11} \in F$ ✓ |

### Para $L_{real}$: cadeia $[s]\ (d^+.d^* \mid d^*.d^+)$

| Forma | Caminho | Final |
|:---|:---|:---|
| $d^+.d^+$ | $\to q_1 \xrightarrow{.} q_5 \xrightarrow{n|0}^+ q_5$ | $q_5 \in F$ ✓ |
| $d^+.$ | $\to q_1 \xrightarrow{.} q_5$ | $q_5 \in F$ ✓ |
| $.d^+$ | $q_0 \xrightarrow{.} q_4 \xrightarrow{n|0} q_5 \xrightarrow{n|0}^* q_5$ | $q_5 \in F$ ✓ |
| $s\ d^+.d^*$ | $\to q_2 \to q_{11} \xrightarrow{.} q_{12} \xrightarrow{n|0}^* q_{12}$ | $q_{12} \in F$ ✓ |

### Para $L_{sci}$: cadeia $r\ (e|E)\ [s]\ d^+$

| Forma base $r$ | Caminho via $e/E$ | Final |
|:---|:---|:---|
| $d^+$ (via $q_1$) | $q_1 \xrightarrow{e} q_{16} \xrightarrow{n|0} q_{17} \xrightarrow{n|0}^* q_{17}$ | $q_{17} \in F$ ✓ |
| $d^+$ com sinal expoente | $q_{16} \xrightarrow{s} q_{18} \xrightarrow{n|0} q_{19} \xrightarrow{n|0}^* q_{19}$ | $q_{19} \in F$ ✓ |
| $d^+.d^*$ (via $q_5$) | $q_5 \xrightarrow{e} q_{20} \xrightarrow{n|0} q_{19}$ | $q_{19} \in F$ ✓ |
| $sd^+$ (via $q_{11}$) | $q_{11} \xrightarrow{e} q_{20} \xrightarrow{s} q_{18} \xrightarrow{n|0} q_{19}$ | $q_{19} \in F$ ✓ |
| qualquer base, expoente com sinal | $q_{20} \xrightarrow{s} q_{18} \xrightarrow{n|0} q_{19}$ | $q_{19} \in F$ ✓ |

### Para $L_{hex}$

| Forma | Caminho | Final |
|:---|:---|:---|
| $0x\ h^+$ | $q_0 \xrightarrow{0} q_3 \xrightarrow{x} q_{13} \xrightarrow{h} q_{14} \xrightarrow{h}^* q_{14}$ | $q_{14} \in F$ ✓ |
| $\#\ h^+$ | $q_0 \xrightarrow{\#} q_{13} \xrightarrow{h} q_{14} \xrightarrow{h}^* q_{14}$ | $q_{14} \in F$ ✓ |
| $h^+h_{sufixo}$ (ex: $FFh$) | $q_0 \xrightarrow{l|e} q_{22} \xrightarrow{h}^* q_{22} \xrightarrow{h} q_{15}$ | $q_{15} \in F$ ✓ |
| $d^+h_{sufixo}$ (ex: $123h$) | $q_0 \to q_1 \xrightarrow{h} q_{15}$ | $q_{15} \in F$ ✓ |

### Para $L_{comp}$: cadeia $a\ s\ b\ i$

| Forma | Caminho | Final |
|:---|:---|:---|
| $d^+ s\ d\ i$ | $\to q_1 \xrightarrow{s} q_6 \xrightarrow{n|0} q_7 \xrightarrow{i} q_{10}$ | $q_{10} \in F$ ✓ |
| $d^+ s\ d^+i$ | $\to q_6 \to q_7 \xrightarrow{n} q_8 \xrightarrow{n|0}^* q_8 \xrightarrow{i} q_{10}$ | $q_{10} \in F$ ✓ |
| $d^+.d^+ s\ d^+.d^+i$ | $\to q_5 \xrightarrow{s} q_6 \to q_7 \to q_8 \xrightarrow{.} q_{21} \xrightarrow{n|0}^+ q_{21} \xrightarrow{i} q_{10}$ | $q_{10} \in F$ ✓ |
| $d\ s\ d.d^+i$ | $\to q_6 \to q_7 \xrightarrow{.} q_9 \xrightarrow{n|0} q_{21} \xrightarrow{i} q_{10}$ | $q_{10} \in F$ ✓ |

---

## 5.5 Verificação de Propriedades Essenciais do AFD

### Propriedade 1: Determinismo

Para todo par $(q, \sigma)$ com $q \in Q$ e $\sigma \in \Sigma_{classe}$, existe **no máximo uma** transição $\delta(q, \sigma)$. A função $\gamma$ é determinística (cada símbolo pertence a exatamente uma classe, com prioridade de aplicação fixa: $n \to l \to e \to s$). Logo $M$ é determinístico. $\square$

### Propriedade 2: Estado de erro absorvente

Para todo $\sigma \in \Sigma_{classe}$: $\delta(q_{err}, \sigma) = q_{err}$ (implícito pelo `dict.get` com default `qerr`). Uma vez em $q_{err}$, o autômato nunca retorna a um estado de aceitação. $\square$

### Propriedade 3: Rejeição de cadeias vazias

A implementação verifica `len(tape) <= 0` antes de processar, retornando `False`. Logo $\varepsilon \notin L(M)$. $\square$

### Propriedade 4: Unicidade do sinal no expoente

A transição $\delta(q_{20}, s) = q_{18}$ leva a $q_{18}$, que **não possui** transição por $s$. Portanto, uma segunda leitura de sinal em $q_{18}$ leva a $q_{err}$. Cadeias como "$1.2e{++}3$" são corretamente rejeitadas. $\square$

---

## 5.6 Conclusão da Prova

$$\boxed{L(M) = L = L_{int} \cup L_{real} \cup L_{sci} \cup L_{hex} \cup L_{comp}}$$

**Soundness** (§5.3): Todo caminho de aceitação termina em um estado final $q_f \in F$ cuja cadeia pertence a alguma $L_i$. Demonstrado por análise exaustiva dos 11 estados finais.

**Completude** (§5.4): Para toda cadeia $w \in L_i$, existe um caminho de $q_0$ até um estado $q_f \in F$. Demonstrado por construção explícita de caminhos para cada forma canônica de cada sublinguagem.

**Determinismo e propriedades auxiliares** (§5.5): $M$ é determinístico, rejeita $\varepsilon$, e garante unicidade de sinal no expoente.

$\therefore$ O autômato $M$ é **correto** com respeito à linguagem $L$. $\blacksquare$

# 6. Conclusão

- Principais aprendizados
- Limitações do estudo

## Principais aprendizados

1. A mistura de 0 como prefixo e número e o E/e como euler e letra de hexadecimal foram os fatores que mais dificultaram o desenvolvimento do autômato

2. Após desenvolver o código principal, trocar o autômato no self.delta foi fácil, possibilitando correções necessárias do processo

3. Números Hexadecimais e números científicos podem ser em boa parte parecidos, sendo diferenciados quando há sinal ou o sufixo h

## Limitações do Estudo

- O autômato apenas identifica se é aceitou ou não, não retorna de que notação aquela sequência pertence
- Para autômatos muito grande o código começa a ficar menos legível por conta do grande mapeamento de transições